# **DEG ANALYSIS**

In [ ]:
# Lock Kaggle's loaded core C-libraries, then install scanpy safely
!pip freeze | grep -E "numpy|scipy|pandas" > constraints.txt
!pip install scanpy -c constraints.txt -q

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import os

sc.settings.verbosity = 2

# ==============================================================================
# 1. Load the Dataset
# ==============================================================================
h5ad_path = "/kaggle/input/datasets/ericdu847/microglial-annotated-multiregion-nuclei-data/SEAAD_Immune_HIP_MEC_LEC_ITG_MTG_FI_STG_DFC_AnG_V1C_RNAseq_final-nuclei.2026-06-22.h5ad"

if not os.path.exists(h5ad_path):
    fallback_paths = [
        "/kaggle/input/microglial-annotated-multiregion-nuclei-data/SEAAD_Immune_HIP_MEC_LEC_ITG_MTG_FI_STG_DFC_AnG_V1C_RNAseq_final-nuclei.2026-06-22.h5ad",
        "/kaggle/input/Microglial annotated multiregion nuclei data/SEAAD_Immune_HIP_MEC_LEC_ITG_MTG_FI_STG_DFC_AnG_V1C_RNAseq_final-nuclei.2026-06-22.h5ad"
    ]
    for alt in fallback_paths:
        if os.path.exists(alt):
            h5ad_path = alt
            break

print(f"Loading dataset from: {h5ad_path}")
adata = sc.read_h5ad(h5ad_path)
print("Data loaded successfully. Shape:", adata.shape)


# ==============================================================================
# 2. Define Groups (Extracted from "Eric data 0731.xlsx")
# ==============================================================================
group_ids = {
    "AD": [
        'H21.33.008', 'H20.33.028', 'H20.33.029', 'H21.33.039', 'H21.33.046', 
        'H20.33.004', 'H20.33.037', 'H20.33.011', 'H21.33.027', 'H21.33.042', 
        'H20.33.018', 'H20.33.020', 'H20.33.026', 'H20.33.031', 'H20.33.045', 
        'H20.33.046', 'H21.33.009', 'H21.33.017', 'H21.33.020', 'H21.33.013', 
        'H20.33.033', 'H21.33.002', 'H21.33.034', 'H21.33.045'
    ],
    "NonAD_AD_patho": [
        'H20.33.001', 'H20.33.002', 'H20.33.008', 'H20.33.012', 'H20.33.013', 
        'H20.33.014', 'H20.33.019', 'H20.33.024', 'H20.33.025', 'H20.33.034', 
        'H20.33.035', 'H20.33.036', 'H20.33.039', 'H20.33.027', 'H20.33.030', 
        'H20.33.044', 'H21.33.003', 'H21.33.004', 'H21.33.011'
    ],
    "nonAD_Ctl": [
        'H21.33.015', 'H21.33.019', 'H21.33.022', 'H21.33.023', 'H21.33.032', 
        'H21.33.033', 'H21.33.036', 'H21.33.037', 'H21.33.041', 'H21.33.047', 
        'H21.33.038'
    ]
}

# Ensure we are using the correct column name for IDs
id_column = 'Donor ID' 
if id_column not in adata.obs.columns:
    print(f"Available columns: {list(adata.obs.columns)}")
    raise ValueError(f"Column '{id_column}' not found in dataset. Please update id_column.")

def assign_condition(sample_id):
    for condition, samples in group_ids.items():
        if sample_id in samples:
            return condition
    return 'Exclude'

# Assign groups to metadata and filter out excluded nuclei
adata.obs['Comparison_Group'] = adata.obs[id_column].apply(assign_condition)
adata_filtered = adata[adata.obs['Comparison_Group'] != 'Exclude'].copy()


# ==============================================================================
# 3. Safely Locate Region Column
# ==============================================================================
region_keywords = ['region', 'roi', 'structure']
region_col = next((col for col in adata_filtered.obs.columns if any(keyword in col.lower() for keyword in region_keywords)), adata_filtered.obs.columns[1])

adata_filtered.obs[region_col] = adata_filtered.obs[region_col].astype(str)
regions = adata_filtered.obs[region_col].dropna().unique()

print(f"Detected Region Column: {region_col}")
print(f"Found {len(regions)} Regions: {', '.join(regions)}\n")


# ==============================================================================
# 4. Define and Run Comparisons Per Region
# ==============================================================================
comparisons = [
    {"target": "AD", "reference": "NonAD_AD_patho", "name": "AD_vs_NonAD_AD_patho"},
    {"target": "AD", "reference": "nonAD_Ctl", "name": "AD_vs_nonAD_Ctl"},
    {"target": "NonAD_AD_patho", "reference": "nonAD_Ctl", "name": "NonAD_AD_patho_vs_nonAD_Ctl"}
]

output_files = []

for comp in comparisons:
    target = comp["target"]
    reference = comp["reference"]
    comp_name = comp["name"]
    
    print(f"\n{'='*60}")
    print(f"STARTING COMPARISON: {comp_name} ({target} vs {reference})")
    print(f"{'='*60}")
    
    for region in regions:
        print(f"\n{'-'*40}")
        print(f"Processing Region: {region}")
        
        # Subset to specific region
        subset = adata_filtered[adata_filtered.obs[region_col] == region].copy()
        
        # Check cell counts for both groups
        counts = subset.obs['Comparison_Group'].value_counts()
        
        if target not in counts or reference not in counts:
            print(f"Skipping {region}: Missing target or reference group entirely.")
            continue
            
        min_cells = 10
        if counts.get(target, 0) < min_cells or counts.get(reference, 0) < min_cells:
            print(f"Skipping {region}: Insufficient nuclei counts ({target}: {counts.get(target, 0)}, {reference}: {counts.get(reference, 0)})")
            continue

        print(f"Nuclei count -> {target}: {counts[target]} | {reference}: {counts[reference]}")
        
        try:
            # Run Wilcoxon rank-sum test
            sc.tl.rank_genes_groups(
                subset, 
                groupby='Comparison_Group', 
                groups=[target], 
                reference=reference, 
                method='wilcoxon',
                use_raw=False 
            )
            
            # Extract results
            result = sc.get.rank_genes_groups_df(subset, group=target)
            
            # Format DataFrame
            formatted_result = pd.DataFrame({
                'Gene': result['names'],
                'LOGFC': result['logfoldchanges'],
                'P-VALUE': result['pvals'],
                'FDR': result['pvals_adj']
            })
            
            # Build safe filename (e.g., HIP_AD_vs_nonAD_Ctl_DGE.csv)
            safe_region = region.replace(" ", "_").replace("/", "_")
            csv_filename = f"{safe_region}_{comp_name}_DGE.csv"
            
            formatted_result.to_csv(csv_filename, index=False)
            output_files.append(csv_filename)
            print(f"Saved: {csv_filename}")
            
        except Exception as e:
            print(f"Error running {region} for {comp_name}: {e}")

print(f"\n{'='*60}")
print(f"Pipeline complete! Successfully generated {len(output_files)} regional CSV files.")